In [1]:
from utils.read_jsonl import read_jsonl
from transformers import BertTokenizer, DistilBertTokenizer
from collections import Counter
import pandas as pd

In [2]:
df_train = read_jsonl("DB-bio/combined_train_and_train_sft_anonymized.jsonl")
df_val = read_jsonl("DB-bio/combined_val_and_val_sft_anonymized.jsonl")
df_combined = pd.concat((df_train, df_val), axis=0, ignore_index=True)
len(df_combined)

4362

In [3]:
df_anonymized = df_combined[df_combined["label"] == 1]
df_not_anonymized = df_combined[df_combined["label"] == 0]
len(df_anonymized), len(df_not_anonymized), len(df_anonymized) + len(df_not_anonymized)

(2181, 2181, 4362)

In [4]:
tokenizer_distilbert = DistilBertTokenizer.from_pretrained("distilbert-base-uncased") 
tokenizer_bert = BertTokenizer.from_pretrained("bert-base-uncased")

In [5]:
def tokenize(row, tok):
    ids = tok(row["text"], padding="max_length", truncation=True, max_length=512)["input_ids"]
    counter_dict = Counter(ids)
    return counter_dict

In [6]:
df_anonymized_counts = df_anonymized.apply(lambda row: tokenize(row, tokenizer_bert), axis=1)
df_not_anonymized_counts = df_not_anonymized.apply(lambda row: tokenize(row, tokenizer_bert), axis=1)

In [7]:
df_anonymized_counts_agg = df_anonymized_counts.sum()
df_not_anonymized_counts_agg = df_not_anonymized_counts.sum()

In [8]:
df_anonymized_tokens_sorted =  sorted(list(df_anonymized_counts_agg.items()), key=lambda x: x[1], reverse=True)
df_not_anonymized_tokens_sorted =  sorted(list(df_not_anonymized_counts_agg.items()), key=lambda x: x[1], reverse=True)
len(df_anonymized_tokens_sorted), len(df_not_anonymized_tokens_sorted)

(13453, 20382)

In [9]:
[(df_anonymized_tokens_sorted[500][1], df_not_anonymized_tokens_sorted[500][1]),
 (df_anonymized_tokens_sorted[1000][1], df_not_anonymized_tokens_sorted[1000][1]),
 (df_anonymized_tokens_sorted[3000][1], df_not_anonymized_tokens_sorted[3000][1])]

[(122, 160), (55, 83), (12, 25)]

In [10]:
all_keys = set(df_anonymized_counts_agg) | set(df_not_anonymized_counts_agg)
comparison = pd.DataFrame(
    [(tokenizer_bert.decode([key]), key, df_not_anonymized_counts_agg.get(key, 0), df_anonymized_counts_agg.get(key, 0)) for key in all_keys],
    columns=['token', 'token_id','original_count', 'anonymized_count']
)
comparison['diff'] = comparison['anonymized_count'] - comparison['original_count']

In [11]:
removed_tokens = comparison.sort_values(by='diff').head(1000)
removed_tokens

,token,token_id,original_count,anonymized_count,diff
130,he,2002,11595,23,-11572
15,",",1010,32280,23933,-8347
138,his,2010,6282,28,-6254
126,and,1998,15926,12232,-3694
36,\,1032,3639,0,-3639
...,...,...,...,...,...
2121,com,4012,106,69,-37
2890,aged,4793,46,9,-37
7008,##wi,9148,38,1,-37
4867,1887,6837,41,4,-37


In [12]:
new_tokens = comparison.sort_values(by='diff', ascending=False).head(1000)
new_tokens

,token,token_id,original_count,anonymized_count,diff
0,[PAD],0,442136,512556,70420
40,a,1037,11129,39388,28259
151,this,2023,565,12776,12211
838,person,2711,46,11036,10990
165,their,2037,355,7934,7579
...,...,...,...,...,...
4108,policies,6043,1,9,8
2252,conducted,4146,15,23,8
7387,performers,9567,8,16,8
10246,automotive,12945,11,19,8


In [13]:
removed_tokens.to_csv("DB-bio/global_analysis_removed_tokens_top_1K.csv", index=False)
new_tokens.to_csv("DB-bio/global_analysis_added_tokens_top_1K.csv", index=False)

In [189]:
tokenizer_bert.decode(list(removed_tokens["token_id"] > ))

'he, his and \\. ( ) was " she the – hers n worldaoe - 1 new himi to st is den marcht york 3 as of three 2 4 americanl john julyer january juney april 2008 may 6 september first august 2012 2010 november 2007 won october\'2009 2013 5 four 2011 at san 2006r k has february 7 co december d sc son 2015 2014 8 all 10 five 2005 / w ’ 12 2004 s : grandon &k la britishmen no 11z or 2002 ko 15 13an open b 16u 2003 9 2000 london father 20es georged six 14 james 17 g c william bo 2001 o chicago 1996in 1998 ga 18v l f washington war brother 22 championships red 1978 1986 1997 cup 21 m 1999 manh 24c prix 1992 e jman be 1974 2016 france i philadelphia 1972 davidg men mo 1969 30 1985 cr t 1994 ma dr al 1963 seven ha wife thomas 19 1993 1989 1970 1976le south 1977 secondel 1995 1988ton 1964 ka los 29 ve hois 25 1980 1991 1990 go 27 best 1984 paulna french 1968 paris mu 1982 26ianone ; br angeles 1983ov 1975 robert 28 eight se 1981 1967 charles himself 1962 australian park 1987 royal old sa german z mi

In [190]:
tokenizer_bert.decode(list(new_tokens["key"]))

'[PAD] a this person their they in year individual city another date early team country major century an late were mid notable 20th several significant multiple from region event town national club prestigious number european various 2000s prominent international certain same period sports 19th player 1990s on 1970s known local them birth with state 1980s professional 1960s well league athlete decade death one have renowned institution top 2010s for 1950s following that figure regional organization born partner different during specific 1940s years other 1930s recognized capital home artist subsequent later company are part political states competitor role tournament competition military midwest global events conflict teams distinguished location parent numerous month horse child sibling historical sport position 1920s high field competitive martial member series few away area passed europe educational large education cycling architect opponent uk religious colleague racing within prev

In [178]:
df_common_tokens = df_anonymized_tokens.join(df_not_anonymized_tokens, how="inner", on="token", lsuffix="_anonym")

In [179]:
len(df_common_tokens)

2361

In [162]:
tokenizer_bert.decode([tup[0] for tup in df_anonymized_tokens_sorted[:50]])

"[PAD] a the in., of this and person to their they was for - with at an on as year's from team ( ) city is individual another major early by date [CLS] [SEP] country born were century one late also that mid national after first"

In [163]:
tokenizer_bert.decode([tup[0] for tup in df_not_anonymized_tokens_sorted[:50]])

'[PAD], the. in and of he a to was - his for ( ) as at\'with s on \\ is " from an by [CLS] [SEP] – borns also first she has one team after world won that new american her who career two which'

In [ ]:
len(df_anonymized_tokens_sorted), len(df_not_anonymized_tokens_sorted)

In [191]:
tokenizer_bert.decode([i for i in range(999, 1200)])

'! " # $ % &\'( ) * +, -. / 0 1 2 3 4 5 6 7 8 9 : ; < = >? @ [ \\ ] ^ _ ` a b c d e f g h i j k l m n o p q r s t u v w x y z { | } ~ ¡ ¢ £ ¤ ¥ ¦ § ¨ © ª « ¬ ® ° ± ² ³ ´ µ ¶ · ¹ º » ¼ ½ ¾ ¿ × ß æ ð ÷ ø þ đ ħ ı ł ŋ œ ƒ ɐ ɑ ɒ ɔ ɕ ə ɛ ɡ ɣ ɨ ɪ ɫ ɬ ɯ ɲ ɴ ɹ ɾ ʀ ʁ ʂ ʃ ʉ ʊ ʋ ʌ ʎ ʐ ʑ ʒ ʔ ʰ ʲ ʳ ʷ ʸ ʻ ʼ ʾ ʿ ˈ ː ˡ ˢ ˣ ˤ α β γ δ ε ζ η θ ι κ λ μ ν ξ ο π ρ ς σ τ υ φ χ ψ ω а б в г д е ж з и к л м н о п р с т у ф'

In [194]:
tokenizer_bert.decode(1012)

'.'